# Astra Training Study
Bucle de entrenamiento espejo de `train.py` (Fisher Sampler + Smart Refinement).

In [1]:
import os, sys
sys.path.append('..')

import torch, numpy as np, matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm

from opensplat3d.utils.train_utils import setup_training, FisherCameraSampler
from opensplat3d.scene import Scene
from opensplat3d.gaussian_model import create_from_pcd
from opensplat3d.gaussian_optimizer import GaussianOptimizer
from opensplat3d.gaussian_renderer import render
from opensplat3d.losses import l1_loss, ssim, instance_2d_loss, get_erank_loss, get_thinness_loss
from opensplat3d.data import load_scene_info
from opensplat3d.utils.vis_utils import pca, enhance_image

device = torch.device('cuda:0')

class Args:
    config = '/home/ubuntu/semantic-gaussians/configs/astra_lab.yaml'
    overrides = []
    wandb = None; checkpoint = None

config = setup_training(Args())
model_params = config.model; opt_params = config.opt; pipe_params = config.pipe

# Carga de escena y cámaras (usará AstraReader automáticamente al detectar el directorio)
scene_info = load_scene_info(model_params)
scene = Scene(scene_info, model_params.resolution, device)
gaussians = create_from_pcd(
    scene_info.point_cloud, 
    model_params.sh_degree, 
    device, 
    with_features=True, 
    feature_dim=model_params.mask_dim
)


# Inicialización del optimizador y sampler
optimizer = GaussianOptimizer(
    gaussians, 
    opt_params, 
    sh_degree=model_params.sh_degree,
    spatial_lr_scale=scene.cameras_extent,
    static_xyz=False,
    device=device
)
sampler = FisherCameraSampler(scene.get_train_cameras())
background = torch.zeros((3), device=device)

print(f"Setup OK. Cámaras cargadas: {len(scene.get_train_cameras())}")
print(f"Puntos iniciales: {gaussians.num_points}")



Using fused SSIM
Found Astra dataset (transforms.json + depth/), using AstraReader!


Loading Astra Training Cameras:   0%|          | 0/239 [00:00<?, ?it/s]

Train: 239 | Test: 0
Loading Training Cameras


  0%|          | 0/239 [00:00<?, ?it/s]

Loading Test Cameras


0it [00:00, ?it/s]

Number of points at initialisation: 1461675
Setup OK. Cámaras cargadas: 239
Puntos iniciales: 1461675


In [2]:
import torch.nn.functional as F
from tqdm import tqdm
import numpy as np

# Definición de PSNR
psnr = lambda img1, img2: 20 * torch.log10(1.0 / torch.sqrt(torch.mean((img1 - img2)**2)))

def depth_to_normal(depth, viewpoint_cam):
    """Calcula normales a partir del mapa de profundidad y la calibración."""
    if depth.ndim == 2: depth = depth.unsqueeze(0).unsqueeze(0)
    elif depth.ndim == 3: depth = depth.unsqueeze(0)
    
    # Derivadas de la profundidad
    dz_dx = torch.zeros_like(depth)
    dz_dy = torch.zeros_like(depth)
    dz_dx[:, :, :, 1:-1] = (depth[:, :, :, 2:] - depth[:, :, :, :-2]) / 2.0
    dz_dy[:, :, 1:-1, :] = (depth[:, :, 2:, :] - depth[:, :, :-2, :]) / 2.0
    
    h, w = depth.shape[-2:]
    fx = viewpoint_cam.image_width / (2 * np.tan(viewpoint_cam.fovX / 2))
    fy = viewpoint_cam.image_height / (2 * np.tan(viewpoint_cam.fovY / 2))
    
    normal = torch.cat([-dz_dx * fx / (depth + 1e-6), 
                        -dz_dy * fy / (depth + 1e-6), 
                        torch.ones_like(depth)], dim=1)
    
    normal = F.normalize(normal, dim=1)
    return normal.squeeze(0)


# --- Bucle de Entrenamiento con Normales GT ---
ITERS = 10000; DISP = 25; losses = []; psnrs = []; ema_loss = 0.0; ema_psnr = 0.0
mask_dim = model_params.mask_dim
lambda_normal = 0.5 # Peso de la pérdida de normales

p_bar = tqdm(range(1, ITERS + 1), desc='Training')
for iteration in p_bar:
    optimizer.update_learning_rate(iteration)
    if iteration % 1000 == 0: optimizer.oneup_sh_degree()
    
    viewpoint_cam = sampler.sample()
    bg = torch.rand(3, device=device) if opt_params.random_background else background
    bg_f = torch.rand(mask_dim, device=device) if opt_params.random_background_features else None
    has_depth = (viewpoint_cam.original_depth is not None)
    
    pkg = render(viewpoint_cam, gaussians, pipe_params, bg, model_params.sh_degree, 
                 optimizer.active_sh_degree, bg_features=bg_f, render_color=True, 
                 render_features=True, render_depth=True, render_var=True)
    
    gt_img = viewpoint_cam.original_image.to(device)
    L1 = l1_loss(pkg.render, gt_img)
    photo_l = (1.0 - opt_params.lambda_dssim) * L1 + opt_params.lambda_dssim * (1.0 - ssim(pkg.render, gt_img))
    loss = opt_params.photo_lambda * photo_l
    
    # 1. Pérdida de Profundidad
    if has_depth:
        gt_depth = viewpoint_cam.original_depth.to(device)

        valid_depth_mask = gt_depth > 0.0

        if valid_depth_mask.any():
            d_l = l1_loss(pkg.depth[valid_depth_mask], gt_depth[valid_depth_mask])
            loss += opt_params.lambda_depth * d_l


        d_l = l1_loss(pkg.depth, viewpoint_cam.original_depth.to(device))
        loss += opt_params.lambda_depth * d_l
    
    # 2. OPTIMIZACIÓN DE NORMALES (Supervisión con GT)
    #rendered_normals = depth_to_normal(pkg.depth, viewpoint_cam)
    #if viewpoint_cam.normal is not None:
        # Calculamos el GT de las normales a partir de la profundidad real
    #    gt_normals = viewpoint_cam.normal.to(device)
        # Supervisión directa
    #    cos_sim = (rendered_normals * gt_normals).sum(dim=0)
    #    loss_normal = (1.0 - torch.abs(cos_sim)).mean()
        
    #    loss += lambda_normal * loss_normal

    # 3. Resto de pérdidas...
    if pkg.variance is not None: loss += opt_params.var_lambda * pkg.variance[:mask_dim].pow(2).mean()
    if viewpoint_cam.masks is not None and iteration % opt_params.inst2d_interval == 0:
        i2d = instance_2d_loss(pkg.features[:mask_dim], viewpoint_cam.masks.to(device), mask_dim, 
                               opt_params.inst2d_sample_size, opt_params.inst2d_gamma, 
                               opt_params.inst2d_weights, opt_params.inst2d_normalize)
        loss += opt_params.inst2d_lambda * i2d['total']
    
    optimizer.optimizer.zero_grad(set_to_none=True); loss.backward()
    
    with torch.no_grad():
        optimizer.add_stats(pkg.viewspace_points, pkg.visibility_filter, pkg.radii, pkg.viewspace_points.grad, None)
        vis = pkg.visibility_filter
        surprise = ((pkg.viewspace_points.grad[vis, :2] ** 2).sum(-1, keepdim=True) / (optimizer.fim_accum[vis] + 0.01)).sum().item()
        sampler.update_score(viewpoint_cam.uid, surprise)
        
        if iteration > opt_params.densify_from_iter and iteration % opt_params.densification_interval == 0:
            optimizer.densify_and_prune(opt_params.densify_grad_threshold, 0.01, scene.cameras_extent, None)
        
        if iteration % opt_params.opacity_reset_interval == 0: optimizer.reset_opacity()
        optimizer.optimizer.step()
        
        # Métricas
        cur_psnr = psnr(pkg.render, gt_img).item()
        losses.append(loss.item()); psnrs.append(cur_psnr)
        ema_loss = 0.1 * loss.item() + 0.9 * ema_loss if iteration > 1 else loss.item()
        ema_psnr = 0.1 * cur_psnr + 0.9 * ema_psnr if iteration > 1 else cur_psnr
    
    if iteration % DISP == 0:
        clear_output(wait=True)
        plt.figure(figsize=(22, 14))
        H, W = gt_img.shape[1], gt_img.shape[2]
        
        def show(pos, img, title, **kw):
            plt.subplot(3,3,pos); plt.imshow(img, **kw); plt.title(title); plt.axis('off')
        
        show(1, pkg.render.detach().permute(1,2,0).cpu().clamp(0,1), f"Render {iteration}")
        show(2, gt_img.permute(1,2,0).cpu().clamp(0,1), "GT RGB")
        
        # Visualización de Normales Renderizadas
        #norm_viz = (rendered_normals.detach().permute(1, 2, 0).cpu().numpy() + 1.0) / 2.0
        #show(3, norm_viz, "Rendered Normals")
        
        d_max = viewpoint_cam.original_depth.max().item() if has_depth else 2.0
        show(4, pkg.depth.detach().cpu().squeeze(), "Depth Render", cmap='plasma', vmin=0, vmax=d_max)
        show(5, viewpoint_cam.original_depth.cpu().squeeze() if has_depth else np.zeros((H,W)), "GT Depth", cmap="plasma", vmin=0, vmax=d_max)
        
        show(6, viewpoint_cam.masks.cpu().numpy()%20 if viewpoint_cam.masks is not None else np.zeros((H,W)), "GT SAM", cmap='tab20')
        
        plt.subplot(3,3,7); plt.plot(losses); plt.yscale('log'); plt.title("Loss History (Log)"); plt.grid(True)
        plt.subplot(3,3,8); plt.plot(psnrs); plt.title("PSNR History"); plt.grid(True)
        
        # Visualización de Normales GT
        #if has_depth:
        #    gt_norm_viz = (gt_normals.detach().permute(1, 2, 0).cpu().numpy() + 1.0) / 2.0
        #    show(9, gt_norm_viz, "GT Normals") # Cambiado a la posición 9 para que esté junto a Semantics
            
        # Semántica (Ahora en la posición 7 u 8 si prefieres moverla)
        # show(9, enhance_image((p_*255).astype(np.uint8)), "Semantics") 
        # (He dejado el espacio para que elijas dónde poner la semántica vs las gráficas)
        
        plt.tight_layout(); plt.show()
        print(f"Iter {iteration} | Loss: {ema_loss:.6f} | PSNR: {ema_psnr:.2f} | Pts: {gaussians.num_points}")

    %matplotlib inline


Training:  39%|███▉      | 3899/10000 [02:54<04:32, 22.40it/s]


KeyboardInterrupt: 